In [5]:
import json
import os
import uuid
from datetime import datetime
from pathlib import Path
from typing import List

from google import genai
from pydantic import BaseModel, Field


# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = "Your-API-key"


MODEL_NAME = "gemini-3.6-flash"

client = genai.Client(api_key=API_KEY)


# Transcript directory
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


# ============================================================
# DATA MODELS
# ============================================================

class InterviewQuestion(BaseModel):
    question_number: int
    question: str
    skill: str
    category: str
    difficulty: str


class QuestionSet(BaseModel):
    questions: List[InterviewQuestion]


class DimensionScores(BaseModel):
    correctness: int = Field(ge=0, le=10)
    relevance: int = Field(ge=0, le=10)
    depth: int = Field(ge=0, le=10)
    reasoning: int = Field(ge=0, le=10)
    practical_application: int = Field(ge=0, le=10)


class AnswerEvaluation(BaseModel):
    question_number: int
    dimension_scores: DimensionScores
    score: int = Field(ge=0, le=10)
    strengths: List[str]
    gaps: List[str]
    feedback: str


class EvaluationSet(BaseModel):
    evaluations: List[AnswerEvaluation]


# ============================================================
# PROMPT 1
# QUESTION GENERATION
# ============================================================

QUESTION_PROMPT = """
You are an expert technical interviewer.

Create a structured mock interview for this role.

ROLE:
{role}

REQUIRED SKILLS:
{skills}

Generate exactly 5 high-quality interview questions.

Requirements:

1. Questions must be directly relevant to the role.
2. Cover the provided skills.
3. Use different question types.
4. Include:
   - fundamentals
   - practical application
   - problem solving
   - scenario-based reasoning
   - technical depth
5. Difficulty should increase gradually.
6. Avoid duplicate questions.
7. Each question must identify the main skill being tested.
8. Each question must have a category.
9. Questions must be answerable using text.
10. Keep questions concise and interview-appropriate.

Difficulty values:
Easy
Medium
Hard

Category values can include:
Fundamentals
Practical Application
Problem Solving
Scenario
Technical Depth

Return exactly 5 questions in the required JSON structure.
"""


# ============================================================
# PROMPT 2
# BATCH ANSWER EVALUATION
# ============================================================

EVALUATION_PROMPT = """
You are a strict and fair technical interviewer.

You are evaluating a completed mock interview.

ROLE:
{role}

REQUIRED SKILLS:
{skills}

Below are the interview questions and candidate answers.

{interview_data}

Evaluate EVERY question.

For each answer, evaluate these five dimensions:

1. Correctness
   Is the technical information accurate?

2. Relevance
   Does the answer directly address the question?

3. Depth
   Does the candidate demonstrate sufficient understanding?

4. Reasoning
   Does the candidate explain how, why, tradeoffs, or logic where needed?

5. Practical Application
   Can the candidate connect the concept to a realistic use case?

Score each dimension from 0 to 10.

Use this general scoring guide:

0-2 = Very poor / incorrect
3-4 = Limited understanding
5-6 = Basic understanding
7-8 = Good understanding
9 = Very strong
10 = Exceptional

For each question:

- Calculate a score from 0-10 based on the five dimensions.
- Give 1-3 specific strengths.
- Give 1-3 specific gaps.
- Give concise professional feedback.

Important:

- Evaluate only what the candidate actually demonstrated.
- Do not invent knowledge or experience.
- Do not reward confidence alone.
- Do not penalize a concise but correct answer.
- Do not evaluate age, gender, personality, accent, or other
  irrelevant characteristics.
- Be technically accurate.
- Be consistent across all five answers.

Return one evaluation for EVERY question.
The question_number must match the supplied question number.
"""


# ============================================================
# GEMINI STRUCTURED REQUEST
# ============================================================

import json
import time


def call_gemini(prompt):
    """Simple Gemini request with retry handling."""

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            if not response.text:
                raise RuntimeError("Gemini returned an empty response.")

            return response.text

        except Exception as error:

            error_text = str(error)

            if (
                "503" in error_text
                or "UNAVAILABLE" in error_text
                or "429" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
            ):
                if attempt < 2:
                    wait = 2 ** attempt
                    print(
                        f"\nGemini temporarily unavailable. "
                        f"Retrying in {wait} seconds..."
                    )
                    time.sleep(wait)
                    continue

            raise RuntimeError(
                f"Gemini API request failed:\n{error}"
            )


# ============================================================
# GENERATE QUESTIONS
# GEMINI CALL #1
# ============================================================

def generate_questions(role, skills):

    prompt = f"""
You are an expert technical interviewer.

Create exactly 5 interview questions for:

Role: {role}

Required skills:
{", ".join(skills)}

Requirements:
- Questions must be directly relevant to the role.
- Cover the required skills.
- Include fundamentals, practical application,
  problem solving, scenario-based reasoning,
  and technical depth.
- Difficulty should increase gradually.
- Do not repeat questions.
- Questions should be suitable for a real technical interview.

Return ONLY valid JSON.

Use exactly this format:

{{
  "questions": [
    {{
      "question_number": 1,
      "question": "question text",
      "skill": "skill being tested",
      "category": "Fundamentals",
      "difficulty": "Easy"
    }}
  ]
}}

Return exactly 5 questions.
"""

    print("\nGenerating 5 role-specific questions...")

    raw_response = call_gemini(prompt)

    
    raw_response = raw_response.strip()

    if raw_response.startswith("```"):
        raw_response = raw_response.replace("```json", "")
        raw_response = raw_response.replace("```", "")
        raw_response = raw_response.strip()

    try:
        data = json.loads(raw_response)

    except json.JSONDecodeError as error:
        raise RuntimeError(
            f"Gemini returned invalid JSON.\n\n"
            f"Response:\n{raw_response}\n\n"
            f"JSON error: {error}"
        )

    questions = data.get("questions", [])

    if len(questions) < 5:
        raise RuntimeError(
            f"Gemini generated only {len(questions)} questions."
        )

    return questions[:3]


# ============================================================
# BATCH EVALUATION
# GEMINI CALL #2
# ============================================================

def evaluate_all_answers(
    role,
    skills,
    questions,
    answers,
):

    interview_data = ""

    for question, answer in zip(
        questions,
        answers,
    ):

        interview_data += f"""
QUESTION {question.question_number}

Skill:
{question.skill}

Category:
{question.category}

Difficulty:
{question.difficulty}

Question:
{question.question}

Candidate Answer:
{answer}

-----------------------------------------
"""

    prompt = EVALUATION_PROMPT.format(
        role=role,
        skills=", ".join(skills),
        interview_data=interview_data,
    )

    print(
        "\nEvaluating all 5 answers in one Gemini request..."
    )

    evaluation_set = call_gemini(
        prompt,
        EvaluationSet,
    )

    

    evaluation_map = {
        evaluation.question_number: evaluation
        for evaluation in evaluation_set.evaluations
    }

    missing = []

    for question in questions:

        if question.question_number not in evaluation_map:

            missing.append(
                question.question_number
            )

    if missing:

        raise RuntimeError(
            "Gemini did not return evaluations for "
            f"questions: {missing}"
        )

    
    return [
        evaluation_map[
            question.question_number
        ]
        for question in questions
    ]


# ============================================================
# PYTHON SCORE CALCULATION
# ============================================================

def calculate_question_score(dimensions):

    score = round(
        (
            dimensions.correctness
            + dimensions.relevance
            + dimensions.depth
            + dimensions.reasoning
            + dimensions.practical_application
        ) / 5
    )

    return score


def calculate_overall_score(evaluations):

    scores = [
        evaluation.score
        for evaluation in evaluations
    ]

    if not scores:

        raise ValueError(
            "No evaluation scores available."
        )

    return round(
        sum(scores) / len(scores),
        2,
    )


# ============================================================
# PERFORMANCE LEVEL
# ============================================================

def get_performance_level(score):

    if score >= 9:

        return "Excellent"

    elif score >= 7:

        return "Strong"

    elif score >= 5:

        return "Moderate"

    else:

        return "Needs Improvement"


# ============================================================
# PYTHON FINAL EVALUATION
# ============================================================

def generate_final_summary(
    role,
    skills,
    questions,
    answers,
    evaluations,
    overall_score,
):
    """
    Generate the final evaluation WITHOUT another Gemini call.

    This keeps the application at exactly two Gemini requests.
    """

    all_strengths = []

    all_gaps = []

    for evaluation in evaluations:

        all_strengths.extend(
            evaluation.strengths
        )

        all_gaps.extend(
            evaluation.gaps
        )

    

    strengths = list(
        dict.fromkeys(all_strengths)
    )

    gaps = list(
        dict.fromkeys(all_gaps)
    )

    
    strengths = strengths[:5]
    gaps = gaps[:5]

    # --------------------------------------------------------
    # Skill performance
    # --------------------------------------------------------

    skill_scores = {}

    for question, evaluation in zip(
        questions,
        evaluations,
    ):

        skill = question.skill

        if skill not in skill_scores:

            skill_scores[skill] = []

        skill_scores[skill].append(
            evaluation.score
        )

    skill_assessment = []

    for skill, scores in skill_scores.items():

        average = round(
            sum(scores) / len(scores),
            1,
        )

        if average >= 8:

            level = "Strong"

        elif average >= 6:

            level = "Moderate"

        else:

            level = "Needs Improvement"

        skill_assessment.append(
            f"{skill}: {average}/10 ({level})"
        )

    # --------------------------------------------------------
    # Recommendation
    # --------------------------------------------------------

    if overall_score >= 8.5:

        recommendation = (
            "The candidate demonstrated strong technical "
            "competence and appears well prepared for the "
            f"{role} role."
        )

    elif overall_score >= 7:

        recommendation = (
            "The candidate demonstrated a good technical "
            "foundation for the role, with some areas that "
            "could be strengthened."
        )

    elif overall_score >= 5:

        recommendation = (
            "The candidate demonstrated basic competence but "
            "would benefit from further development in the "
            "identified skill gaps."
        )

    else:

        recommendation = (
            "The candidate demonstrated significant gaps "
            "relative to the skills evaluated in this interview."
        )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    summary = (
        f"The candidate completed a five-question interview "
        f"for the {role} role and achieved an overall score "
        f"of {overall_score}/10. "
    )

    if strengths:

        summary += (
            "The strongest demonstrated areas were: "
            + "; ".join(strengths[:3])
            + ". "
        )

    if gaps:

        summary += (
            "The main areas for improvement were: "
            + "; ".join(gaps[:3])
            + "."
        )

    return {
        "performance_level": get_performance_level(
            overall_score
        ),
        "strengths": strengths,
        "gaps": gaps,
        "skill_assessment": skill_assessment,
        "recommendation": recommendation,
        "summary": summary,
    }


# ============================================================
# TRANSCRIPT CREATION
# ============================================================

def create_transcript(
    interview_id,
    timestamp,
    role,
    skills,
    questions,
    answers,
    evaluations,
    overall_score,
    final_evaluation,
):

    question_results = []

    for question, answer, evaluation in zip(
        questions,
        answers,
        evaluations,
    ):

        question_results.append(
            {
                "question_number":
                    question.question_number,

                "question":
                    question.question,

                "skill":
                    question.skill,

                "category":
                    question.category,

                "difficulty":
                    question.difficulty,

                "candidate_answer":
                    answer,

                "dimension_scores":
                    evaluation.dimension_scores.model_dump(),

                "score":
                    evaluation.score,

                "strengths":
                    evaluation.strengths,

                "gaps":
                    evaluation.gaps,

                "feedback":
                    evaluation.feedback,
            }
        )

    return {
        "interview_id": interview_id,
        "timestamp": timestamp,
        "role": role,
        "skills": skills,
        "number_of_questions": 5,
        "questions": question_results,
        "overall_score": overall_score,
        "performance_level":
            final_evaluation["performance_level"],
        "final_evaluation":
            final_evaluation,
    }


# ============================================================
# SAVE TRANSCRIPT
# ============================================================

def save_transcript(transcript):

    file_path = DATA_DIR / (
        f"interview_{transcript['interview_id']}.json"
    )

    with open(
        file_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            transcript,
            file,
            indent=4,
            ensure_ascii=False,
        )

    return file_path


# ============================================================
# DISPLAY QUESTION
# ============================================================

def display_question(
    question,
    current,
    total,
):

    print("\n" + "=" * 75)

    print(
        f"QUESTION {current}/{total}"
    )

    print("=" * 75)

    print(
        f"\nSkill: {question.skill}"
    )

    print(
        f"Category: {question.category}"
    )

    print(
        f"Difficulty: {question.difficulty}"
    )

    print(
        f"\n{question.question}"
    )


# ============================================================
# GET MULTI-LINE ANSWER
# ============================================================

def get_answer():

    print(
        "\nYour answer "
        "(press Enter twice when finished):"
    )

    lines = []

    while True:

        line = input()

        if line.strip() == "":
            break

        lines.append(line)

    return "\n".join(lines).strip()


# ============================================================
# DISPLAY FINAL RESULTS
# ============================================================

def display_final_results(
    questions,
    evaluations,
    overall_score,
    final_evaluation,
    transcript_path,
):

    print("\n")
    print("=" * 75)

    print("INTERVIEW RESULTS")

    print("=" * 75)

    print(
        f"\nOverall Score: "
        f"{overall_score}/10"
    )

    print(
        f"Performance Level: "
        f"{final_evaluation['performance_level']}"
    )

    # --------------------------------------------------------
    # Question scores
    # --------------------------------------------------------

    print("\nQuestion Scores:")

    for question, evaluation in zip(
        questions,
        evaluations,
    ):

        print(
            f"  Q{question.question_number}: "
            f"{evaluation.score}/10"
        )

    # --------------------------------------------------------
    # Strengths
    # --------------------------------------------------------

    print("\nStrengths:")

    for strength in final_evaluation["strengths"]:

        print(
            f"  - {strength}"
        )

    # --------------------------------------------------------
    # Gaps
    # --------------------------------------------------------

    print("\nGaps:")

    for gap in final_evaluation["gaps"]:

        print(
            f"  - {gap}"
        )

    # --------------------------------------------------------
    # Skill assessment
    # --------------------------------------------------------

    print("\nSkill Assessment:")

    for assessment in final_evaluation[
        "skill_assessment"
    ]:

        print(
            f"  - {assessment}"
        )

    # --------------------------------------------------------
    # Recommendation
    # --------------------------------------------------------

    print("\nRecommendation:")

    print(
        f"  {final_evaluation['recommendation']}"
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print("\nSummary:")

    print(
        f"  {final_evaluation['summary']}"
    )

    # --------------------------------------------------------
    # Transcript
    # --------------------------------------------------------

    print("\nTranscript:")

    print(
        f"  Saved to: {transcript_path}"
    )

    print("\n" + "=" * 75)


# ============================================================
# MAIN INTERVIEW
# ============================================================

def run_interview():

    print("\n" + "=" * 75)

    print("AI INTERVIEW AGENT")

    print(
        "Role-specific structured technical interview"
    )

    print("=" * 75)

    # --------------------------------------------------------
    # ROLE
    # --------------------------------------------------------

    while True:

        role = input(
            "\nEnter the job role: "
        ).strip()

        if role:
            break

        print(
            "Role cannot be empty."
        )

    # --------------------------------------------------------
    # SKILLS
    # --------------------------------------------------------

    while True:

        raw_skills = input(
            "Enter required skills "
            "(comma-separated): "
        ).strip()

        skills = [
            skill.strip()
            for skill in raw_skills.split(",")
            if skill.strip()
        ]

        # Remove duplicate skills.
        skills = list(
            dict.fromkeys(skills)
        )

        if skills:
            break

        print(
            "Please enter at least one skill."
        )

    # --------------------------------------------------------
    # CONFIGURATION
    # --------------------------------------------------------

    print("\n" + "=" * 75)

    print("INTERVIEW CONFIGURATION")

    print("=" * 75)

    print(
        f"Role: {role}"
    )

    print(
        f"Skills: {', '.join(skills)}"
    )

    print(
        "Questions: 5"
    )

    # --------------------------------------------------------
    # GEMINI CALL #1
    # --------------------------------------------------------

    questions = generate_questions(
        role,
        skills,
    )

    print(
        "\nSuccessfully generated "
        f"{len(questions)} questions."
    )

    # --------------------------------------------------------
    # COLLECT ALL ANSWERS
    # --------------------------------------------------------

    answers = []

    for index, question in enumerate(
        questions,
        start=1,
    ):

        display_question(
            question,
            index,
            len(questions),
        )

        answer = get_answer()

        while not answer:

            print(
                "\nAnswer cannot be empty."
            )

            answer = get_answer()

        answers.append(answer)

        print(
            "\nAnswer recorded."
        )

    # --------------------------------------------------------
    # GEMINI CALL #2
    # --------------------------------------------------------

    evaluations = evaluate_all_answers(
        role,
        skills,
        questions,
        answers,
    )

    # --------------------------------------------------------
    # CORRECT SCORE CALCULATION
    # --------------------------------------------------------

    # Recalculate each question score in Python
    # from the five dimensions returned by Gemini.

    for evaluation in evaluations:

        evaluation.score = calculate_question_score(
            evaluation.dimension_scores
        )

    # --------------------------------------------------------
    # OVERALL SCORE
    # --------------------------------------------------------

    overall_score = calculate_overall_score(
        evaluations
    )

    # --------------------------------------------------------
    # FINAL EVALUATION
    # --------------------------------------------------------

    final_evaluation = generate_final_summary(
        role=role,
        skills=skills,
        questions=questions,
        answers=answers,
        evaluations=evaluations,
        overall_score=overall_score,
    )

    # --------------------------------------------------------
    # TRANSCRIPT
    # --------------------------------------------------------

    interview_id = str(
        uuid.uuid4()
    )

    timestamp = datetime.now().isoformat()

    transcript = create_transcript(
        interview_id=interview_id,
        timestamp=timestamp,
        role=role,
        skills=skills,
        questions=questions,
        answers=answers,
        evaluations=evaluations,
        overall_score=overall_score,
        final_evaluation=final_evaluation,
    )

    transcript_path = save_transcript(
        transcript
    )

    # --------------------------------------------------------
    # DISPLAY
    # --------------------------------------------------------

    display_final_results(
        questions=questions,
        evaluations=evaluations,
        overall_score=overall_score,
        final_evaluation=final_evaluation,
        transcript_path=transcript_path,
    )

    return transcript


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    try:

        run_interview()

    except KeyboardInterrupt:

        print(
            "\n\nInterview cancelled by user."
        )

    except Exception as error:

        
        print(
            "\n\nERROR:"
        )

        print(error)

        print(
            "\nPlease check your Gemini API configuration "
            "and try again."
        )


AI INTERVIEW AGENT
Role-specific structured technical interview

Enter the job role: Data Analysis
Enter required skills (comma-separated): Python

INTERVIEW CONFIGURATION
Role: Data Analysis
Skills: Python
Questions: 5

Generating 5 role-specific questions...

Gemini temporarily unavailable. Retrying in 1 seconds...

Gemini temporarily unavailable. Retrying in 2 seconds...


ERROR:
Gemini API request failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Please check your Gemini API configuration and try again.
